# 🔬 PUF Fotónica SmartLight — Cuaderno de Laboratorio
**TFM · Oxel Arnal Martínez · ITEAM/PRL · UPV · 2025**

---
Ejecuta las celdas **en orden**, de arriba a abajo.

Cuando cambies el setup físico del laboratorio para medir un nuevo puerto,
cambia `INPORT` en la **Celda 1** y vuelve a ejecutar desde la **Celda 6**.

| Celda | Qué hace | ¿Cuántas veces? |
|-------|----------|-----------------|
| 0 | Imports y funciones auxiliares | 1 vez |
| 1 | Parámetros y rutas | 1 vez |
| 2 | Conectar chip y láser | 1 vez |
| 3 | Fases pasivas φ₀ | 1 vez |
| 4 | TBUs dinámicas y canales MEDA | 1 vez |
| 5 | Generar retos (corrientes mA) | 1 vez |
| 6 | Verificar ruta del puerto actual | Cada puerto |
| 7 | **MEDIR — bucle de retos** | Cada puerto |
| 8 | Comprobar reproducibilidad | Cada puerto |
| 9 | Calcular métricas PUF | Al final |
| 10 | Desconectar el chip | Al final |

## Celda 0 — Imports y funciones auxiliares
Ejecutar una sola vez al abrir el cuaderno.

In [ ]:
import json, math, random, hashlib, time, pathlib
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from smartlight.smartlight import Smartlight

# ── Función: firma Lehmer-Gray ────────────────────────────────────────────
def lehmer_gray_signature(powers_dict, ports):
    """
    Convierte un vector de potencias en una firma binaria.
      1. Ordena los puertos por potencia descendente → permutación
      2. Calcula el código de Lehmer de esa permutación
      3. Codifica cada dígito Lehmer en código Gray
    Para 27 puertos de salida → ~3052 bits.
    """
    N = len(ports)
    sorted_ports = sorted(ports, key=lambda p: powers_dict.get(p, 0.0), reverse=True)
    perm         = [ports.index(p) for p in sorted_ports]
    used, lehmer = [False] * N, []
    for val in perm:
        lehmer.append(sum(1 for j in range(val) if not used[j]))
        used[val] = True
    bits = []
    for i, digit in enumerate(lehmer):
        max_val = N - i
        n_bits  = max(1, math.ceil(math.log2(max_val + 1))) if max_val > 1 else 1
        gray    = digit ^ (digit >> 1)
        for b in range(n_bits - 1, -1, -1):
            bits.append((gray >> b) & 1)
    return bits

def hamming_distance(s1, s2):
    n = min(len(s1), len(s2))
    return sum(a != b for a, b in zip(s1[:n], s2[:n])) / n if n > 0 else 0.0

def sha256_bits(bits):
    n   = len(bits)
    val = int(''.join(str(b) for b in bits), 2)
    raw = val.to_bytes((n + 7) // 8, 'big')
    return [int(x) for byte in hashlib.sha256(raw).digest() for x in f'{byte:08b}']

print('✓ Imports y funciones cargados.')

## Celda 1 — Parámetros y rutas
Ajusta los valores de esta celda antes de empezar.

> **Cuando cambies de puerto:** modifica solo `INPORT` y vuelve a ejecutar desde la Celda 6.

In [ ]:
# ── Ajustar antes de cada sesión ─────────────────────────────────────────
CONFIG_PATH      = pathlib.Path('config.json')  # fichero del chip
LASER_POWER_DBM  = -3.0    # confirmar con el técnico
LASER_WAVELENGTH = 1550.0  # confirmar con el técnico (nm)
I_MAX_MA         = 20.0    # corriente máxima por heater — confirmar con el técnico
N_CHALLENGES     = 200     # retos por puerto
SEED             = 42      # no cambiar entre sesiones

# ── Puerto de entrada activo ──────────────────────────────────────────────
# Cambia este número cada vez que cambies el setup físico del laboratorio.
INPORT = 0

# ── Puertos físicamente accesibles (los índices 22-33 están bloqueados) ───
ACTIVE_PORTS = list(range(0, 22)) + list(range(34, 40))  # 28 puertos

# ── Directorio de resultados ──────────────────────────────────────────────
RESULTS_DIR = pathlib.Path('resultados')
RESULTS_DIR.mkdir(exist_ok=True)

# ── Rutas maestras del Anillo Isotrópico ─────────────────────────────────
# Para cada puerto: lista de (ID_TBU, estado)
#   estado = 'x' (CROSS) | '=' (BAR) | float (coupling factor)
RUTAS = {
    0:  [(0,'x'),(6,'x'),(10,'x'),(16,'x'),(21,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    1:  [(0,'='),(6,'x'),(10,'x'),(16,'x'),(21,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    2:  [(9,'x'),(10,'='),(16,'x'),(21,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    3:  [(15,'x'),(19,'='),(20,'x'),(21,'='),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    4:  [(15,'x'),(9,'='),(10,'='),(16,'x'),(21,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    5:  [(19,'x'),(20,'x'),(21,'='),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    6:  [(28,'x'),(29,'x'),
         (30,'x'),(31,1-1/6),(36,1-1/5),(41,1-1/4),(40,1-1/3),(35,1-1/2)],
    7:  [(34,'x'),(38,'='),(39,'x'),
         (35,'x'),(30,1-1/6),(31,1-1/5),(36,1-1/4),(41,1-1/3),(40,1-1/2)],
    8:  [(34,'x'),(28,'='),(29,'x'),
         (30,'x'),(31,1-1/6),(36,1-1/5),(41,1-1/4),(40,1-1/3),(35,1-1/2)],
    9:  [(38,'x'),(39,'x'),
         (35,'x'),(30,1-1/6),(31,1-1/5),(36,1-1/4),(41,1-1/3),(40,1-1/2)],
    10: [(47,'='),(44,'x'),(39,'x'),
         (35,'x'),(30,1-1/6),(31,1-1/5),(36,1-1/4),(41,1-1/3),(40,1-1/2)],
    11: [(53,'x'),(57,'='),(58,'='),(54,'x'),(49,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    12: [(53,'x'),(47,'='),(48,'x'),(49,'='),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    13: [(57,'x'),(58,'='),(54,'x'),(49,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    14: [(66,'='),(63,'x'),(58,'x'),(54,'x'),(49,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    15: [(66,'x'),(63,'x'),(58,'x'),(54,'x'),(49,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    16: [(67,'x'),(63,'='),(58,'x'),(54,'x'),(49,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    17: [(68,'x'),(64,'='),(59,'x'),(54,'='),(49,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    18: [(69,'x'),(64,'x'),(59,'x'),(54,'='),(49,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    19: [(70,'x'),(65,'='),(61,'x'),(55,'x'),(50,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    20: [(71,'x'),(65,'x'),(61,'x'),(55,'x'),(50,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    21: [(71,'='),(65,'x'),(61,'x'),(55,'x'),(50,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    22: [(62,'x'),(61,'='),(55,'x'),(50,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    23: [(56,'x'),(52,'='),(51,'x'),(50,'='),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    24: [(56,'x'),(62,'='),(61,'='),(55,'x'),(50,'x'),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    25: [(52,'x'),(51,'x'),(50,'='),(45,'x'),
         (40,'x'),(35,1-1/6),(30,1-1/5),(31,1-1/4),(36,1-1/3),(41,1-1/2)],
    26: [(43,'x'),(42,'x'),
         (41,'x'),(40,1-1/6),(35,1-1/5),(30,1-1/4),(31,1-1/3),(36,1-1/2)],
    27: [(37,'x'),(33,'='),(32,'x'),
         (36,'x'),(41,1-1/6),(40,1-1/5),(35,1-1/4),(30,1-1/3),(31,1-1/2)],
    28: [(37,'x'),(43,'='),(42,'x'),
         (41,'x'),(40,1-1/6),(35,1-1/5),(30,1-1/4),(31,1-1/3),(36,1-1/2)],
    29: [(33,'x'),(32,'x'),
         (36,'x'),(41,1-1/6),(40,1-1/5),(35,1-1/4),(30,1-1/3),(31,1-1/2)],
    30: [(24,'='),(27,'x'),(32,'x'),
         (36,'x'),(41,1-1/6),(40,1-1/5),(35,1-1/4),(30,1-1/3),(31,1-1/2)],
    31: [(18,'x'),(14,'='),(13,'='),(17,'x'),(22,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    32: [(18,'x'),(24,'x'),(27,'x'),(32,'x'),
         (36,'x'),(41,1-1/6),(40,1-1/5),(35,1-1/4),(30,1-1/3),(31,1-1/2)],
    33: [(14,'x'),(13,'='),(17,'x'),(22,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    34: [(5,'='),(8,'x'),(13,'x'),(17,'x'),(22,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    35: [(5,'x'),(8,'x'),(13,'x'),(17,'x'),(22,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    36: [(4,'x'),(8,'='),(13,'x'),(17,'x'),(22,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    37: [(3,'x'),(7,'='),(12,'x'),(17,'='),(22,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    38: [(2,'x'),(7,'='),(11,'x'),(16,'='),(21,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
    39: [(1,'x'),(6,'='),(10,'x'),(16,'x'),(21,'x'),(26,'x'),
         (31,'x'),(36,1-1/6),(41,1-1/5),(40,1-1/4),(35,1-1/3),(30,1-1/2)],
}

ALL_ROUTE_PUCS = sorted({tbu for ruta in RUTAS.values() for (tbu, _) in ruta})
outports       = [p for p in ACTIVE_PORTS if p != INPORT]

print(f'Puerto activo:      {INPORT}')
print(f'Puertos de salida:  {len(outports)}')
print(f'TBUs en rutas:      {len(ALL_ROUTE_PUCS)}')
print(f'Resultados en:      {RESULTS_DIR}/')
print(f'\nRuta del puerto {INPORT}:')
for tbu, estado in RUTAS[INPORT]:
    if isinstance(estado, str):
        print(f'  TBU {tbu:>3}  {"CROSS" if estado == "x" else "BAR":>6}  (gateway)')
    else:
        print(f'  TBU {tbu:>3}  {estado:>6.4f}  (anillo)')

## Celda 2 — Conectar el chip y configurar el láser
Ejecutar **una sola vez** al inicio de la sesión.

> `reset_mesh()` pone todos los PUCs a corriente 0 → estado limpio, solo φ₀ activa.

In [ ]:
sl = Smartlight(filepath=CONFIG_PATH, simulation=False)
print(f'Chip conectado  |  Simulación: {sl.get_sim_status()}')

if not sl.mesh_calibrated():
    print('Cargando calibración...')
    sl.calibration()
print(f'Calibración: {"OK" if sl.mesh_calibrated() else "FALLO"}')

# Estado limpio: todos los PUCs a corriente 0
# → cada TBU opera solo bajo su fase pasiva de fabricación φ₀
sl.reset_mesh()
print('reset_mesh() — todos los PUCs en φ₀ puro')

# Configurar láser
laser = sl.get_laser()
laser.laser_activation(activate=True)
laser.set_output_power(output_power=LASER_POWER_DBM)
laser.set_laser_wavelength(wavelength=LASER_WAVELENGTH)
print(f'Láser: ON  |  {LASER_POWER_DBM} dBm  |  {LASER_WAVELENGTH} nm')

## Celda 3 — Leer las fases pasivas del chip (φ₀)
Las fases pasivas son el **ADN de fabricación** del chip. Son únicas e irrepetibles.
Las guardamos para referencia pero **no las usamos para calibrar nada**.

In [ ]:
passive_phases = sl.get_puc_passive_phases()
# passive_phases = {puc_id: fase_en_radianes}

print(f'Fases pasivas leídas: {len(passive_phases)} PUCs')
print()
print(f'{"PUC":>5}  {"φ₀ (rad)":>10}')
print('-' * 20)
for puc_id, phi in sorted(passive_phases.items()):
    print(f'{puc_id:>5}  {phi:>10.4f}')

with open(RESULTS_DIR / 'passive_phases.json', 'w') as f:
    json.dump({str(k): v for k, v in passive_phases.items()}, f, indent=2)
print(f'\nGuardado en {RESULTS_DIR}/passive_phases.json')

## Celda 4 — TBUs dinámicas y canales MEDA
Las **TBUs dinámicas** son las que nunca se calibran — su respuesta lleva el fingerprint del chip.

El MEDA controla los heaters por **canales**, no por PUC IDs.
Cada TBU tiene 2 heaters (brazo up y brazo down) → 2 canales por TBU.

In [ ]:
# Todos los PUCs del chip
all_pucs = sl.get_active_pucs_mesh()
print(f'PUCs totales en el chip: {len(all_pucs)}')
print(f'IDs: {sorted(all_pucs)}')

# TBUs dinámicas = las que NO están en ninguna ruta
dynamic_pucs = sorted(set(all_pucs) - set(ALL_ROUTE_PUCS))
print(f'\nTBUs de ruta (gateway+anillo): {len(ALL_ROUTE_PUCS)}')
print(f'TBUs dinámicas (entropía PUF): {len(dynamic_pucs)}')
print(f'IDs dinámicas: {dynamic_pucs}')

# Mapa de canales MEDA
meda     = sl.get_meda()
wma      = sl.get_wma()
full_map = meda.get_mapping()   # {puc_id: [canal_up, canal_down]}

channel_map = {puc: full_map[puc] for puc in dynamic_pucs if puc in full_map}
missing     = [puc for puc in dynamic_pucs if puc not in full_map]
if missing:
    print(f'\nAVISO: TBUs sin mapeo MEDA (se excluyen): {missing}')

print(f'\nMapeo MEDA de TBUs dinámicas:')
print(f'{"PUC":>5}  {"canal_up":>9}  {"canal_down":>10}')
print('-' * 30)
for puc, (cu, cd) in channel_map.items():
    print(f'{puc:>5}  {cu:>9}  {cd:>10}')

all_dynamic_channels = [ch for canales in channel_map.values() for ch in canales]
print(f'\nCanales dinámicos totales: {len(all_dynamic_channels)} '
      f'({len(channel_map)} TBUs × 2 heaters)')

## Celda 5 — Generar los retos
Un reto = `{canal_meda: corriente_mA}` para todos los canales dinámicos.

Las corrientes se aplican **directamente al MEDA sin ninguna conversión**.
La librería no calcula fases — manda los mA al driver del heater.

$$\varphi_{\text{total}} = \varphi_0 + \Delta\varphi(I_{\text{reto}})$$

El `SEED` fijo garantiza que los retos son siempre los mismos números.

In [ ]:
rng        = random.Random(SEED)
challenges = [
    {canal: round(rng.uniform(0.0, I_MAX_MA), 3) for canal in all_dynamic_channels}
    for _ in range(N_CHALLENGES)
]

print(f'Retos generados:     {len(challenges)}')
print(f'Canales por reto:    {len(challenges[0])}')
print(f'Rango de corrientes: [0.0, {I_MAX_MA}] mA')
print()
print('Ejemplo — reto 0 (primeros 6 canales):')
print(f'{"Canal":>7}  {"Corriente (mA)":>15}')
print('-' * 26)
for canal, mA in list(challenges[0].items())[:6]:
    print(f'{canal:>7}  {mA:>15.3f}')

with open(RESULTS_DIR / 'challenges.json', 'w') as f:
    json.dump([{str(k): v for k, v in ch.items()} for ch in challenges], f, indent=2)
print(f'\nRetos guardados en {RESULTS_DIR}/challenges.json')

## Celda 6 — Verificar la ruta del puerto actual
Comprueba que la fibra está bien acoplada y la ruta funciona.
Ejecutar **cada vez que cambies INPORT**.

In [ ]:
print(f'Verificando ruta del puerto {INPORT}...')
print()

# Programar solo la ruta (sin reto — TBUs dinámicas en φ₀ puro)
sl.reset_puc(pucs_id=ALL_ROUTE_PUCS)
sl.interconnect(puc_state_list=RUTAS[INPORT])

powers_test = sl.get_output_power(inport=INPORT, outport=outports)
total_dBm   = sum(powers_test.values())

print(f'Potencia total en salidas: {total_dBm:.2f} dBm')
print()
print(f'{"Puerto":>7}  {"Potencia (dBm)":>15}')
print('-' * 25)
for puerto, pot in sorted(powers_test.items(), key=lambda x: x[1], reverse=True):
    barra = '█' * max(0, int((pot + 40) / 2))
    print(f'{puerto:>7}  {pot:>10.2f}   {barra}')

print()
if total_dBm < -60.0:
    print('⚠  POTENCIA MUY BAJA — revisar acoplamiento de fibra')
else:
    print(f'✓  Puerto {INPORT} listo para medir')

## Celda 7 — Medir los retos  ⬅ celda principal
Ejecutar **una vez por cada puerto**. El progreso se muestra en tiempo real.

**Ciclo de cada reto:**
1. `reset_puc(ALL_ROUTE_PUCS)` — limpia gateway y anillo
2. `interconnect(RUTAS[INPORT])` — programa la ruta
3. `set_multichannel_current(reto)` — corrientes crudas a TBUs dinámicas
4. `get_output_power()` — lee la distribución de potencia
5. `lehmer_gray_signature()` — binariza a firma de ~3052 bits

In [ ]:
signatures  = []
powers_list = []
t0          = time.time()

# Barra de progreso con ipywidgets
barra    = widgets.IntProgress(value=0, min=0, max=N_CHALLENGES,
                               description='Midiendo:', bar_style='info',
                               layout=widgets.Layout(width='500px'))
etiqueta = widgets.Label(value=f'0 / {N_CHALLENGES}')
display(widgets.HBox([barra, etiqueta]))

for idx, reto in enumerate(challenges):

    # 1. Limpiar TBUs de ruta
    sl.reset_puc(pucs_id=ALL_ROUTE_PUCS)

    # 2. Programar gateway + anillo para este puerto
    sl.interconnect(puc_state_list=RUTAS[INPORT])

    # 3. Corrientes crudas en TBUs dinámicas  →  φ_total = φ₀ + Δφ(I)
    meda.set_multichannel_current(core=wma, currents_dict=reto)

    # 4. Leer potencias de salida
    powers = sl.get_output_power(inport=INPORT, outport=outports)

    # 5. Binarizar con Lehmer-Gray
    sig = lehmer_gray_signature(powers, outports)

    signatures.append(sig)
    powers_list.append({str(k): v for k, v in powers.items()})

    # Actualizar barra de progreso
    barra.value    = idx + 1
    elapsed        = time.time() - t0
    remaining      = (elapsed / (idx + 1)) * (N_CHALLENGES - idx - 1)
    etiqueta.value = (f'{idx+1} / {N_CHALLENGES}  |  '
                      f'{elapsed:.0f}s  |  ~{remaining:.0f}s restantes')

barra.bar_style = 'success'
t_total = time.time() - t0
print(f'\n✓ {len(signatures)} retos medidos en {t_total:.1f}s  '
      f'({t_total/N_CHALLENGES:.2f}s por reto)')
print(f'  Longitud de cada firma: {len(signatures[0])} bits')

# Guardar resultados del puerto
fname = f'port_{INPORT:02d}.json'
with open(RESULTS_DIR / fname, 'w') as f:
    json.dump({'inport': INPORT, 'outports': outports,
               'sig_length': len(signatures[0]), 'elapsed_s': round(t_total, 2),
               'signatures': signatures, 'powers': powers_list}, f, indent=2)
print(f'  Guardado en {RESULTS_DIR}/{fname}')

# Mostrar qué puertos faltan
medidos    = sorted([int(p.stem.split('_')[1]) for p in RESULTS_DIR.glob('port_*.json')])
pendientes = [p for p in ACTIVE_PORTS if p not in medidos]
print(f'\nPuertos medidos:    {medidos}')
print(f'Puertos pendientes: {pendientes}')
if pendientes:
    print(f'\n→ Cambia INPORT = {pendientes[0]} en la Celda 1 y ejecuta desde la Celda 6')

## Celda 8 — Comprobar reproducibilidad
Repite el mismo reto 10 veces y mide la HD entre las firmas.
Debe ser ≈ 0. Si es > 0.05 el chip no está estable.

In [ ]:
N_REP = 10
sigs_rep = []

print(f'Repitiendo reto 0 — {N_REP} veces...')
for rep in range(N_REP):
    sl.reset_puc(pucs_id=ALL_ROUTE_PUCS)
    sl.interconnect(puc_state_list=RUTAS[INPORT])
    meda.set_multichannel_current(core=wma, currents_dict=challenges[0])
    p = sl.get_output_power(inport=INPORT, outport=outports)
    sigs_rep.append(lehmer_gray_signature(p, outports))
    print(f'  Rep {rep+1:2d}/{N_REP}', end='\r')

ref     = sigs_rep[0]
hd_vals = [hamming_distance(ref, s) for s in sigs_rep[1:]]
hd_r    = sum(hd_vals) / len(hd_vals)

print(f'\n{"Rep":>5}  {"HD vs rep 1":>12}')
print('-' * 20)
for i, hd in enumerate(hd_vals, start=2):
    print(f'{i:>5}  {hd:>12.4f}')

print(f'\nHD_r medio: {hd_r:.4f}  (umbral: < 0.05)')
if hd_r < 0.05:
    print('✓ Chip estable')
else:
    print('⚠ Inestabilidad detectada — espera a que se estabilice la temperatura')

## Celda 9 — Calcular métricas PUF
Ejecutar cuando hayas medido todos los puertos que quieras.
Lee todos los ficheros `port_XX.json` y calcula las métricas pre y post SHA-256.

In [ ]:
port_files = sorted(RESULTS_DIR.glob('port_*.json'))
print(f'Puertos encontrados: {[f.stem for f in port_files]}')
print()

todas_sigs = []
for pf in port_files:
    with open(pf) as f:
        d = json.load(f)
    todas_sigs.extend(d['signatures'])
    print(f'  {pf.stem}: {len(d["signatures"])} firmas  |  {d["sig_length"]} bits')

print(f'\nFirmas totales: {len(todas_sigs)}')

# Independencia
hd_pairs = [hamming_distance(todas_sigs[i], todas_sigs[j])
             for i in range(len(todas_sigs))
             for j in range(i+1, min(i+21, len(todas_sigs)))]
independencia = sum(hd_pairs) / len(hd_pairs)

# Uniformidad
uniformidad = sum(sum(s)/len(s) for s in todas_sigs) / len(todas_sigs)

# Bit aliasing
min_len = min(len(s) for s in todas_sigs)
mat     = np.array([s[:min_len] for s in todas_sigs], dtype=float)
ba      = float(np.mean(np.abs(mat.mean(axis=0) - 0.5)))

# Post-SHA256
sigs_hash      = [sha256_bits(s) for s in todas_sigs]
unif_hash      = sum(sum(s)/len(s) for s in sigs_hash) / len(sigs_hash)
mat_h          = np.array(sigs_hash, dtype=float)
ba_hash        = float(np.mean(np.abs(mat_h.mean(axis=0) - 0.5)))
hd_hash_pairs  = [hamming_distance(sigs_hash[i], sigs_hash[j])
                  for i in range(len(sigs_hash))
                  for j in range(i+1, min(i+21, len(sigs_hash)))]
indep_hash     = sum(hd_hash_pairs) / len(hd_hash_pairs)

print()
print(f'{"Métrica":<30} {"Medido":>10} {"GT simulado":>12}')
print('=' * 55)
print(f'{"[Pre-hash]":<30}')
print(f'{"  Independencia HD":<30} {independencia:>10.4f} {0.4418:>12.4f}')
print(f'{"  Uniformidad":<30} {uniformidad:>10.4f} {0.4671:>12.4f}')
print(f'{"  Bit aliasing":<30} {ba:>10.4f} {0.1257:>12.4f}')
print(f'{"  Longitud firma (bits)":<30} {min_len:>10}  {3052:>11}')
print(f'{"[Post-SHA256]":<30}')
print(f'{"  Independencia HD":<30} {indep_hash:>10.4f} {0.5005:>12.4f}')
print(f'{"  Uniformidad":<30} {unif_hash:>10.4f} {0.5000:>12.4f}')
print(f'{"  Bit aliasing":<30} {ba_hash:>10.4f} {0.0000:>12.4f}')

metricas = {
    'puertos': [f.stem for f in port_files], 'firmas': len(todas_sigs),
    'longitud_bits': min_len, 'independencia': independencia,
    'uniformidad': uniformidad, 'bit_aliasing': ba,
    'independencia_hash': indep_hash, 'uniformidad_hash': unif_hash,
    'bit_aliasing_hash': ba_hash,
}
with open(RESULTS_DIR / 'metricas.json', 'w') as f:
    json.dump(metricas, f, indent=2)
print(f'\nGuardado en {RESULTS_DIR}/metricas.json')

## Celda 10 — Desconectar el chip
Ejecutar **siempre al terminar**, aunque haya habido errores.

In [ ]:
sl.reset_mesh()   # todos los PUCs a corriente 0
sl.disconnect()   # desconectar subsistemas
print('✓ Chip desconectado correctamente.')